# 🤖 Build Your Own RAG Chatbot from Scratch


## A Beginner-Friendly Guide (Zero to Working Chatbot)

    Welcome! Today, you are going to build a chatbot that doesn't just "guess" answers, 
    but actually reads documents to give you facts.

## 📚 What is a "Normal" Chatbot?

    A normal AI (like ChatGPT) is like a student who memorized a million books years ago 
    but isn't allowed to look at new notes. 
    If you ask it about something that happened yesterday or a private document you wrote, 
    it will hallucinate (confidently lie).

## 🔍 What is a RAG Chatbot?

    RAG stands for Retrieval-Augmented Generation.
    Think of it as giving the AI an "Open Book Exam." 
        1. Retrieval: The AI looks through your specific documents to find the right page.
        2. Augmented: It adds that information to your question.
        3. Generation: It writes a perfect answer based only on those facts.

### Flowchart of RAG

```mermaid
flowchart TD
    Q[User Question] --> EmbedQ[Embed];

    DB[Vector DB] -->|top-k| Retrieve[Retrieve relevant chunks];

    EmbedQ --> Retrieve;
    Retrieve --> Context[Augmented Prompt = query + chunks];

    Context --> LLM(LLM);
    LLM -->|generates| R[Answer];

In [2]:
# STEP 1: INSTALLING OUR TOOLS
# We need to download some 'libraries' (pre-written code) to help us.

# 'langchain' is the main framework for building AI apps
!pip install langchain langchain-community langchain-huggingface 

# 'chromadb' is our "Vector Database" (the AI's library where it stores facts)
!pip install chromadb 

# 'sentence-transformers' helps the AI understand the "meaning" of sentences
!pip install sentence-transformers 

# 'pypdf' lets the AI read PDF files
!pip install pypdf 

!pip install openai

!pip install PyMuPDF
print("✅ All tools installed successfully!")

  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.2.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Using cached huggingface_hub-1.6.0-py3-none-any.whl.metadata (13 kB)
Using cached huggingface_hub-1.6.0-py3-none-any.whl (612 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into ac

## 🧪 Experiment 1: The "Normal" AI (No Context)

In [24]:

from openai import OpenAI

# --- 1. THE CONNECTION (Setup) ---
# We use the OpenAI library but point the 'base_url' to OpenRouter.
# This allows us to access dozens of different AI models with one piece of code.
api_key = "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"
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    )



# --- 2. THE TARGET (The Question) ---
# This is the specific mystery we are trying to solve using the technical manual.
question = """
For the F/A-18 Hornet, below what aircraft gross weight does the Flight Control System (FCS)
g-limiter permit a maximum commanded symmetric positive load factor (NzREF) of 7.5 g?
"""



# --- 3. THE PERSONA (The Expert) ---
# This "System" message sets the AI's professional boundaries and tone.
# It ensures the AI talks like an engineer, not a chatbot.
role = """
You are AeroBot – a no-nonsense aerospace & aviation expert.

Rules:
- Answer short, direct and to the point
- Use as few words as possible
- Never explain unless explicitly asked "explain" or "why"
- No introductions, no conclusions, no apologies, no filler phrases
- Only talk about aerospace, aircraft, airlines, aerodynamics, propulsion, avionics, spaceflight, airports, pilots, incidents – nothing else
- If question is unrelated → reply only: "Off-topic. Aerospace/plane question only."

Examples of good answers:
User: What is the cruise speed of A320?
You: Mach 0.78 (~828 km/h)

User: Why do 737 MAX have those split wingtips?
You: Increases aspect ratio, reduces induced drag, improves fuel efficiency ~3-4%

User: Tell me about yourself
You: Off-topic. Aerospace/plane question only.
"""


# --- 5. THE EXECUTION (The API Call) ---
# We package everything up and send it to the model.
# The 'system' role is the personality; the 'user' role is the specific task.
response = client.chat.completions.create(
        # Using a free model for testing
        model="openrouter/free",
        messages=[
            {"role": "system", "content": role},
            {"role": "user", "content": question}
            ]
        )
        
# --- 6. THE DELIVERY (Display Results) ---
# Pull the text content out of the AI's response package and print it.
print("🤖 AI Says:")
print(response.choices[0].message.content)

🤖 AI Says:
35,000 lb gross weight.



### Analysis (For comparison !)

#### From NATOPS FLIGHT MANUAL NAVY MODEL F/A-18A/B/C/D 161353 AND UP AIRCRAFT
    
    11.1.7 G−Limiter. The g−limiter function in the FCS limits commanded load factor under most
           flight conditions to the symmetric load limit (NzREF) based on gross weight below 
           44,000 pounds gross weight (maximum NzREF limit of 7.5 g). Above 44,000 pounds, 
           NzREF is held constant at 5.5 g. The negative load factor limit command is fixed at 
           negative 3 g’s for all gross weights. At weights greater than 32,357 pounds the 
           g−limiter does not provide adequate negative g protection and aircraft overstress 
           is possible.

    Correct ? Yes/no ?

## 🧪 Experiment 2: "Prompt Stuffing" (The Messy Way)

    Now, let's try to give the AI the information by "stuffing" it all into the prompt.
    
        The Problem: If your document is 1,000 pages long, you can't fit it in the 
        chat box. It's too expensive and the AI gets "lost in the middle."

### Read a PDF

In [23]:
import fitz  # PyMuPDF: The library used to "read" and parse PDF files

# The file path is the specific location of your technical manual
pdf_path = "F18-ABCD-000.pdf"

# --- 1. ACCESS THE DOCUMENT ---
# Open the file so we can begin scanning its contents
doc = fitz.open(pdf_path)
print(f"Success! Opened a PDF with {len(doc)} pages\n")

# --- 2. INITIALIZE THE "COLLECTION" ---
# This variable will hold all the text we extract from every page.
# Think of it as a single, long digital transcript.
PDF_content = ""

# --- 3. THE EXTRACTION LOOP ---
# We visit every page one-by-one, from the first to the last.
for i in range(len(doc)):
    page = doc[i]  # "Pick up" the current page
    
    # OCR/Text Scanning: Convert the visual layout into readable strings.
    text = page.get_text("text") 

    # --- 4. FORMATTING THE TRANSCRIPT ---
    # We add visual dividers (═) so the AI (and we) can see where pages start and end.
    # This acts like a "table of contents" built directly into the text.
    PDF_content += f"\n{'═' * 80}\n"
    PDF_content += f"Page {i+1:3d} of {len(doc)}\n"
    PDF_content += f"{'═' * 80}\n\n"

    # --- 5. VALIDATION ---
    # We check if the page actually contains text.
    # If a page is just a photo or diagram, the "text" will be empty.
    if text.strip():
        PDF_content += text + "\n\n"
    else:
        PDF_content += "[No extractable text on this page - likely an image or diagram]\n\n"

# --- 6. FINALIZING ---
# Close the file to free up system memory.
doc.close()

# The 'PDF_content' variable is now ready to be sent to an AI or a database!
print("Extraction complete. Text is ready for processing.")

Success! Opened a PDF with 902 pages

Extraction complete. Text is ready for processing.


### Pass to LLM

    We take the full text of the PDF and 'stuff' it directly into the instructions we give the AI. 
    It’s like saying: 'Hey AI, read this entire 50-page document first, then answer my one small 
    question at the very bottom.

In [10]:
from openai import OpenAI

# --- 1. THE CONNECTION (Setup) ---
# We use the OpenAI library but point the 'base_url' to OpenRouter.
# This allows us to access dozens of different AI models with one piece of code.
api_key = "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"
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    )


# --- 2. THE TARGET (The Question) ---
# This is the specific mystery we are trying to solve using the technical manual.
# We are asking about the F/A-18 Hornet's "G-limiter." 
# Think of a G-limiter as the car's speed limiter, but for how hard a plane can turn.
question = """
For the F/A-18 Hornet (legacy A/B/C/D variants), below what aircraft gross weight does the 
Flight Control System (FCS) g-limiter permit a maximum commanded symmetric positive 
load factor (NzREF) of 7.5 g?
"""

# --- 3. THE TEMPLATE (The "Open-Book Exam") ---
# We use an 'f-string' (the f before the quotes) to plug in our PDF text.
# This tells the AI: "Here is a massive pile of data (PDF). Look ONLY here."
prompt = f"""
You are a precise technical assistant. Answer ONLY using the provided CONTEXT.

QUESTION:
{question}

CONTEXT:
{PDF}

Rules — you MUST follow all of them:
1. Use ONLY facts explicitly stated in the CONTEXT. (No guessing!)
2. Do NOT use outside knowledge. (Ignore what you learned on the internet.)
3. If the answer isn't in the text, say: "Not enough information."
4. Keep it short, factual, and use bullet points.
"""

# --- 4. THE PERSONA (The Expert) ---
# This "System" message sets the AI's professional boundaries and tone.
# It ensures the AI talks like an engineer, not a chatbot.
role = """
You are AeroBot – a no-nonsense aerospace & aviation expert.

Rules:
- Answer short, direct and to the point
- Use as few words as possible
- Never explain unless explicitly asked "explain" or "why"
- No introductions, no conclusions, no apologies, no filler phrases
- Only talk about aerospace, aircraft, airlines, aerodynamics, propulsion, avionics, spaceflight, airports, pilots, incidents – nothing else
- If question is unrelated → reply only: "Off-topic. Aerospace/plane question only."

Examples of good answers:
User: What is the cruise speed of A320?
You: Mach 0.78 (~828 km/h)

User: Why do 737 MAX have those split wingtips?
You: Increases aspect ratio, reduces induced drag, improves fuel efficiency ~3-4%

User: Tell me about yourself
You: Off-topic. Aerospace/plane question only.
"""

# --- 5. THE EXECUTION (The API Call) ---
# We package everything up and send it to the model.
# The 'system' role is the personality; the 'user' role is the specific task.
response = client.chat.completions.create(
        # Using a free model for testing
        model="openrouter/free",
        messages=[
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
            ]
        )
        
# --- 6. THE DELIVERY (Display Results) ---
# Pull the text content out of the AI's response package and print it.
print("🤖 AI Says:")
print(response.choices[0].message.content)

BadRequestError: Error code: 400 - {'error': {'message': 'This endpoint\'s maximum context length is 262144 tokens. However, you requested about 615959 tokens (615959 of text input). Please reduce the length of either one, or use the "middle-out" transform to compress your prompt automatically.', 'code': 400, 'metadata': {'provider_name': None}}}

#### Analysis

    while this is easy, it has some "side effects":
        The "TL;DR" Problem: Just like humans, if a prompt is too long, the AI might get "distracted" or 
        forget the details in the middle (this is actually called Lost in the Middle).

        The Cost: In the real world, AI companies charge by the word. Stuffing a whole PDF into every 
        question is like paying for a whole pizza when you only wanted one pepperoni.

        The Limit: Every AI has a "storage limit" for a single conversation (the Context Window). If the
        PDF is a massive 500-page novel, it might not even fit!

## 🏗️ The 6-Stage RAG Pipeline

    RAG is the "Smart Way." Instead of sending the whole book, we only send the relevant paragraph.

    Step	
        1. Load	Read your PDF/Text	
        2. Split	Cut text into small pieces	
        3. Embed	Turn text into "Meaning Numbers"	
        4. Store	Put numbers in a Vector DB	
        5. Retrieve	Find chunks that match the question	
        6. Generate	AI writes the final answer	

### Load,Split,Embed and Store

    Why do we need "Vectors"?
        Imagine you search for "Jet Engine."
            A normal search looks for the exact letters J-E-T.
            A Vector search understands that "Turbine" or "Propulsion" are mathematically similar to "Jet Engine."

In [21]:
import fitz  # PyMuPDF for reading PDFs
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# --- 1. THE ARCHITECTURE (The Setup) ---
pdf_path = "F18-ABCD-000.pdf"
persist_directory = "./chroma_db"  # Folder where our "digital brain" lives

# The Translator (Embeddings): Converts human language into math (vectors) 
# so the computer can understand the "meaning" of sentences.
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# The Slicer (Text Splitter): Breaks long pages into 1000-character chunks.
# The 100-character "overlap" ensures sentences don't get cut in half.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# --- 2. THE FACTORY LINE (Extract & Process) ---
doc = fitz.open(pdf_path)
print(f"Opened PDF with {len(doc)} pages\n")

all_page_documents = []

for i in range(len(doc)):
    page = doc[i]
    text = page.get_text("text")

    if text.strip():
        # Chop the page into manageable, bite-sized pieces
        page_chunks = text_splitter.split_text(text)
        
        # Wrap each piece in a "Document" object with a digital 'sticky note' (metadata)
        # telling us exactly which page it came from.
        for chunk in page_chunks:
            new_doc = Document(
                page_content=chunk,
                metadata={"page": i + 1, "source": pdf_path}
            )
            all_page_documents.append(new_doc)
    else:
        print(f"Skipping page {i+1}: Likely an image or empty page.")

doc.close()

# --- 3. THE ARCHIVE (Creating the Vector Database) ---
# This is the heavy lifting: turning text into math and saving it to disk.
print(f"Encoding {len(all_page_documents)} chunks into the database...")



vector_db = Chroma.from_documents(
    documents=all_page_documents,
    embedding=embeddings,
    persist_directory=persist_directory
)

print("Success! Your searchable digital brain is saved in './chroma_db'.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Opened PDF with 902 pages

Skipping page 4: Likely an image or empty page.
Skipping page 6: Likely an image or empty page.
Skipping page 34: Likely an image or empty page.
Skipping page 36: Likely an image or empty page.
Skipping page 63: Likely an image or empty page.
Skipping page 66: Likely an image or empty page.
Skipping page 72: Likely an image or empty page.
Skipping page 284: Likely an image or empty page.
Skipping page 298: Likely an image or empty page.
Skipping page 302: Likely an image or empty page.
Skipping page 306: Likely an image or empty page.
Skipping page 346: Likely an image or empty page.
Skipping page 382: Likely an image or empty page.
Skipping page 432: Likely an image or empty page.
Skipping page 466: Likely an image or empty page.
Skipping page 468: Likely an image or empty page.
Skipping page 538: Likely an image or empty page.
Skipping page 604: Likely an image or empty page.
Skipping page 626: Likely an image or empty page.
Skipping page 666: Likely an ima

### Retrieve

    Now, we give those search results to the AI like an open-book reference. Instead of guessing from memory, 
    the AI reads these specific notes to write a factual answer for the user.

In [22]:
# --- 1. THE SEARCH QUERY ---
# Define the specific technical question we want to answer.
query = """
        For the F/A-18 Hornet (legacy A/B/C/D variants), below what aircraft gross weight does the Flight Control System (FCS) 
        g-limiter permit a maximum commanded symmetric positive load factor (NzREF) of 7.5 g?
        """

# --- 2. VECTOR SEARCH (Finding Context) ---
# Search the database for the 3 most relevant text snippets (k=3).
# Instead of matching keywords, it matches the "meaning" of the query.
docs = vector_db.similarity_search(query, k=3)

# --- 3. RESULTS INSPECTION ---
# Loop through the matches to verify the information found in the PDF.
for d in docs:
    print(f"--- Result Found ---")
    # Display the actual text found in the document
    print(f"Content: {d.page_content}...")
    
    # Display the source metadata (e.g., the specific page number)
    print(f"Source: Page {d.metadata['page']}\n")

--- Result Found ---
Content: 11.1.7
G−Limiter.
The g−limiter function in the FCS limits commanded load factor under most
flight conditions to the symmetric load limit (NzREF) based on gross weight below 44,000 pounds gross
weight (maximum NzREF limit of 7.5 g). Above 44,000 pounds, NzREF is held constant at 5.5 g. The
negative load factor limit command is fixed at negative 3 g’s for all gross weights. At weights greater
than 32,357 pounds the g−limiter does not provide adequate negative g protection and aircraft
overstress is possible.
During rolling maneuvers, the g−limiter reduces commanded load factor up to 80% NzREF. The
additional reduction begins with 0.75 inch lateral stick up to 80% at full lateral stick input of 3 inches.
Very abrupt stick commands can exceed the capabilities of the system and result in an overstress. The
g−limiter can be disengaged during emergency situations by pressing the paddle switch to allow 33%...
Source: Page 436

--- Result Found ---
Content: 11.1.7

### Generate

In [19]:
from openai import OpenAI

# --- CONFIGURATION ---
# Connect to OpenRouter using the OpenAI library
api_key = "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"
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

# --- 1. THE DATA QUERY ---
# The specific technical question we want the AI to answer.
question = """
    For the F/A-18 Hornet (legacy A/B/C/D variants), below what aircraft gross weight does the Flight Control System (FCS) 
    g-limiter permit a maximum commanded symmetric positive load factor (NzREF) of 7.5 g?
    """

# --- 2. THE PROMPT TEMPLATE ---
# This "wraps" our question with strict instructions and the source data (docs).
# It forces the AI to act like an 'Open Book' exam taker.
prompt = f"""
    You are a precise technical assistant. Answer ONLY using the provided CONTEXT.

    QUESTION:
    {question}

    CONTEXT:
    {docs}

    Rules:
    1. Use ONLY facts from the CONTEXT. (No guessing!)
    2. No "common knowledge."
    3. If the answer is missing, say: "Not enough information."
    4. Use bullet points and keep units exactly as they appear in the manual.
    """

# --- 3. THE SYSTEM PERSONA ---
# Defines the AI's personality: a strict, "no-filler" aerospace expert.
role = """
You are AeroBot – a no-nonsense aerospace & aviation expert.

Rules:
- Answer short, direct and to the point
- Use as few words as possible
- Never explain unless explicitly asked "explain" or "why"
- No introductions, no conclusions, no apologies, no filler phrases
- Only talk about aerospace, aircraft, airlines, aerodynamics, propulsion, avionics, spaceflight, airports, pilots, incidents – nothing else
- If question is unrelated → reply only: "Off-topic. Aerospace/plane question only."

Examples of good answers:
User: What is the cruise speed of A320?
You: Mach 0.78 (~828 km/h)

User: Why do 737 MAX have those split wingtips?
You: Increases aspect ratio, reduces induced drag, improves fuel efficiency ~3-4%

User: Tell me about yourself
You: Off-topic. Aerospace/plane question only.
"""

# --- 4. THE API CALL ---
# Send the persona (system) and the instructions (user) to the model.
response = client.chat.completions.create(
        model="openrouter/free", # Using a free model for testing
        messages=[
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ]
    )

# --- 5. THE OUTPUT ---
# Extract and display the final answer from the AI's response.
print("--- ENGINEER'S REPORT ---")
print(response.choices[0].message.content)

--- ENGINEER'S REPORT ---
- 7.5 g symmetric positive load factor (NzREF) is permitted below 44,000 pounds gross weight.

